# Cell commands and magics ✨

Lines starting with `!` run shell commands and lines starting with `%` are
magics (`%help` lists them all). Most follow the special commands of
[GoNB](https://github.com/janpfeifer/gonb), the Go kernel for Jupyter. Cell
magics like `%%writefile` or `%%sh` take the whole cell. Files are written in
the directory programs run in: this notebook uses a `gopyter-demo` directory
and removes it at the end.

In [ ]:
%version

In [ ]:
// Shell commands run in the kernel's workspace, a Go module.
!go version
!ls

## Files and scripts

In [ ]:
%%writefile gopyter-demo/languages.csv
language,year,typing
Go,2009,static
Python,1991,dynamic
Rust,2015,static
JavaScript,1995,dynamic
Zig,2016,static

In [ ]:
import "encoding/csv"

// readCSV reads a CSV file, without its header.
func readCSV(path string) [][]string {
	f, err := os.Open(path)
	if err != nil {
		panic(err)
	}
	defer f.Close()
	rows, err := csv.NewReader(f).ReadAll()
	if err != nil {
		panic(err)
	}
	return rows[1:]
}

// The program runs in the same directory, so it finds the file.
languages := readCSV("gopyter-demo/languages.csv")
languages

In [ ]:
%%sh
echo "statically typed, newest first:"
awk -F, '$3 == "static" { print $2, $1 }' gopyter-demo/languages.csv | sort -r

In [ ]:
%%script awk '{ total += $2; print } END { print "total:", total }'
apples 3
pears 5
figs 12

## Environment, arguments and flags

In [ ]:
%env GREETING=hello
fmt.Println(os.Getenv("GREETING"), "from %env")

In [ ]:
import "flag"

var who = flag.String("who", "world", "who to greet")
var times = flag.Int("n", 1, "how many times")

In [ ]:
%% -who=gopher -n=2 extra args
// As in GoNB, %% parses the flags first, here with this cell's arguments.
for range *times {
	fmt.Println("Hello,", *who)
}
fmt.Println("the rest:", flag.Args())

In [ ]:
%exec greet -who=everyone
// %exec runs a function instead of statements, after parsing the flags.
func greet() { fmt.Printf("Greetings, %s!\n", *who) }

## Tests and benchmarks

In [ ]:
// Reverse reverses a string by runes, so it works for any UTF-8 text.
func Reverse(s string) string {
	r := []rune(s)
	slices.Reverse(r)
	return string(r)
}

In [ ]:
%test
// %test builds the cell with go test and runs its tests and benchmarks.
func TestReverse(t *testing.T) {
	for in, want := range map[string]string{"": "", "go": "og", "héllo, 世界": "界世 ,olléh"} {
		if got := Reverse(in); got != want {
			t.Errorf("Reverse(%q) = %q, want %q", in, got, want)
		}
	}
}

func BenchmarkReverse(b *testing.B) {
	for b.Loop() {
		Reverse("the quick brown fox jumps over the lazy dog")
	}
}

## Build flags and capturing output

In [ ]:
%goflags "-ldflags=-X main.build=demo-build"
// %goflags passes flags to go build, here to set a variable at link time.
var build = "dev"
fmt.Println("build:", build)

In [ ]:
%goflags ""
// Cleared: back to the default.
fmt.Println("build:", build)

In [ ]:
%capture gopyter-demo/report.txt
// %capture also writes this cell's output to a file.
fmt.Println("the answer is", 6*7)

In [ ]:
report, err := os.ReadFile("gopyter-demo/report.txt")
fmt.Printf("%q %v\n", report, err)

## Declarations

In [ ]:
// %ls lists what the notebook has declared so far; %rm forgets names.
%ls

In [ ]:
%%sh
rm -r gopyter-demo && echo "cleaned up gopyter-demo"